<a href="https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/reasoning/visualizing_reasoning_traces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing Reasoning Traces from Mistral Models

_Authored by: Côme Rossary, Stanford University ([@comersy](https://github.com/comersy))_

In this tutorial, you will learn how to turn a model's chain-of-thought into a structured, visual representation that is easy to read and audit.

When a reasoning model thinks through a problem, its trace is often long, free-form, and hard to follow. The trace contains hypotheses, deductions, verifications, and sometimes backtracks — but presented as plain text, this structure is invisible.

> Mistral offers two ways to obtain reasoning traces: **native reasoning models** ([Magistral](https://docs.mistral.ai/capabilities/reasoning/native)) which always expose their thinking in a dedicated field, and **adjustable reasoning** ([`mistral-small-latest`](https://docs.mistral.ai/capabilities/reasoning/adjustable)) which lets you control reasoning per request via `reasoning_effort`. This notebook uses the latter — available on the free tier — and prompts the model to label each reasoning step explicitly.

We will follow these steps:

* Generate a reasoning trace by asking `mistral-small-latest` with `reasoning_effort='high'` to think step by step.
* Extract the structure from the trace by calling the model again to return a JSON array of typed steps with dependencies.
* Visualize the result as a table and an interactive graph.

We run the full pipeline on three problems: a **math** problem, a **code** problem, and an **abstract** reasoning problem.

## Getting Started

We will install [`pyvis`](https://pyvis.readthedocs.io), a Python wrapper around `vis.js` that renders interactive network graphs in notebooks. Everything else (`requests`, `matplotlib`) is already available in Colab.

In [ ]:
!pip install pyvis -q

Now we import the dependencies and set up the API client. We use the Mistral REST API directly via `requests` — `reasoning_effort` is supported as a top-level parameter on the chat completions endpoint.

> Get your free API key at [console.mistral.ai](https://console.mistral.ai/api-keys/). In Colab, open the **Secrets** panel (the key icon on the left sidebar) and add a secret named `MISTRAL_API_KEY`.

In [ ]:
import os
import re
import json
import time
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML, display

# In Colab: open Secrets panel (key icon, left sidebar)
# and add a secret named MISTRAL_API_KEY
try:
    from google.colab import userdata
    api_key = userdata.get('MISTRAL_API_KEY')
except Exception:
    api_key = os.environ.get('MISTRAL_API_KEY')

if not api_key:
    raise ValueError('MISTRAL_API_KEY not found.')

API_URL = 'https://api.mistral.ai/v1/chat/completions'
HEADERS  = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type':  'application/json',
}

print('Ready.')

## Building the pipeline

The pipeline has three pieces:

* `call_and_reason(problem)` — calls `mistral-small-latest` with `reasoning_effort='high'` and a system prompt that asks the model to label each step.
* `extract_steps(trace)` — calls the model again with a JSON extraction prompt, returning a list of typed steps with their dependencies.
* `print_steps(steps)` and `draw_graph(steps)` — render the structured output as an HTML table and an interactive `pyvis` graph.

Each reasoning step is classified as one of six types: **observation**, **hypothesis**, **deduction**, **verification**, **backtrack**, or **conclusion**. This taxonomy lets us color and connect the steps in the visualization.

In [ ]:
import re
import json
from IPython.display import HTML, display

# ── Prompts ───────────────────────────────────────────────────

REASONING_SYSTEM_PROMPT = """You are a careful reasoner. When given a problem, think through it step by step.
Format your reasoning as a numbered list where each step starts with its type in brackets:
[observation], [hypothesis], [deduction], [verification], [backtrack], or [conclusion].

Example format:
1. [observation] The function returns `a` instead of `b` at the end.
2. [hypothesis] The loop may be running one iteration short.
3. [deduction] For n=1, range(0) runs 0 times, so a stays 0 instead of becoming 1.
4. [conclusion] The fix is to return `b` instead of `a`.

After the reasoning steps, provide a clear final answer separated by a line '---'."""

EXTRACTION_PROMPT = """You are given a reasoning trace formatted as a numbered list.
Extract each step and return a JSON array.

Each object must have:
- "id": integer (step number)
- "type": one of observation/hypothesis/deduction/verification/backtrack/conclusion
- "summary": one sentence, max 20 words
- "depends_on": list of step ids this step follows from (empty if it's a starting step)

Return ONLY valid JSON. No explanation, no markdown fences.

Reasoning trace:
{trace}
"""

# ── API calls ─────────────────────────────────────────────────

def call_and_reason(problem):
    resp = requests.post(API_URL, headers=HEADERS, json={
        'model':            'mistral-small-latest',
        'reasoning_effort': 'high',
        'messages': [
            {'role': 'system', 'content': REASONING_SYSTEM_PROMPT},
            {'role': 'user',   'content': problem},
        ],
    })
    data = resp.json()
    if 'choices' not in data:
        raise ValueError(data.get('message', data))

    raw = data['choices'][0]['message']['content']
    if isinstance(raw, list):
        content = ' '.join(c.get('text', '') for c in raw if isinstance(c, dict))
    else:
        content = raw

    parts = content.split('---', 1)
    return {
        'trace':  parts[0].strip(),
        'answer': parts[1].strip() if len(parts) > 1 else '',
        'usage':  data['usage'],
    }


def extract_steps(trace):
    prompt = EXTRACTION_PROMPT.format(trace=trace[:6000])
    resp = requests.post(API_URL, headers=HEADERS, json={
        'model':    'mistral-small-latest',
        'messages': [{'role': 'user', 'content': prompt}],
    })
    data = resp.json()
    if 'choices' not in data:
        raise ValueError(data.get('message', data))
    raw = data['choices'][0]['message']['content']
    raw = re.sub(r'```json|```', '', raw).strip()
    return json.loads(raw)


# ── Visualization: Cytoscape.js interactive graph ─────────────

TYPE_COLORS = {
    'observation':  '#3498db',
    'hypothesis':   '#2ecc71',
    'deduction':    '#f1c40f',
    'verification': '#e67e22',
    'backtrack':    '#e74c3c',
    'conclusion':   '#9b59b6',
}

def print_steps(steps):
    """Display steps as a clean HTML table."""
    rows = ''
    for s in steps:
        color = TYPE_COLORS.get(s['type'], '#95a5a6')
        deps  = ', '.join(str(d) for d in s.get('depends_on', [])) or '—'
        rows += f"""
        <tr>
          <td style="padding:10px 14px;border-bottom:1px solid #eee;font-weight:600;color:#333">{s['id']}</td>
          <td style="padding:10px 14px;border-bottom:1px solid #eee">
            <span style="background:{color};color:white;padding:3px 10px;border-radius:12px;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:0.3px">{s['type']}</span>
          </td>
          <td style="padding:10px 14px;border-bottom:1px solid #eee;color:#666;font-size:13px">{deps}</td>
          <td style="padding:10px 14px;border-bottom:1px solid #eee;color:#222">{s['summary']}</td>
        </tr>"""

    html = f"""
    <div style="font-family:-apple-system,sans-serif;margin:16px 0">
      <table style="border-collapse:collapse;width:100%;background:white;border-radius:8px;overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,0.08)">
        <thead>
          <tr style="background:#2c3e50;color:white">
            <th style="padding:12px 14px;text-align:left;font-weight:600;font-size:13px">#</th>
            <th style="padding:12px 14px;text-align:left;font-weight:600;font-size:13px">Type</th>
            <th style="padding:12px 14px;text-align:left;font-weight:600;font-size:13px">Depends on</th>
            <th style="padding:12px 14px;text-align:left;font-weight:600;font-size:13px">Summary</th>
          </tr>
        </thead>
        <tbody>{rows}</tbody>
      </table>
    </div>"""
    display(HTML(html))

def draw_graph(steps, title='Reasoning Graph'):
    if not steps:
        print('No steps to draw.')
        return

    net = Network(
        height='600px',
        width='100%',
        directed=True,
        bgcolor='#fafafa',
        font_color='#222',
        notebook=True,
        cdn_resources='in_line',
    )

    # Hierarchical layout (top to bottom)
    net.set_options("""
    {
      "layout": {
        "hierarchical": {
          "enabled": true,
          "direction": "UD",
          "sortMethod": "directed",
          "levelSeparation": 130,
          "nodeSpacing": 200,
          "treeSpacing": 200
        }
      },
      "physics": {
        "enabled": false
      },
      "edges": {
        "smooth": {
          "type": "cubicBezier",
          "forceDirection": "vertical",
          "roundness": 0.4
        },
        "arrows": {
          "to": {"enabled": true, "scaleFactor": 0.7}
        },
        "color": {"color": "#999"},
        "width": 1.5
      },
      "nodes": {
        "shape": "box",
        "borderWidth": 2,
        "margin": 12,
        "font": {"size": 13, "face": "arial", "color": "#222"},
        "widthConstraint": {"minimum": 180, "maximum": 220}
      },
      "interaction": {
        "hover": true,
        "dragNodes": true,
        "zoomView": true
      }
    }
    """)

    # Add nodes
    for s in steps:
        color = TYPE_COLORS.get(s['type'], '#95a5a6')
        label = f"#{s['id']}  {s['type'].upper()}\n\n{s['summary']}"
        title_html = f"<b>{s['type'].upper()}</b><br>{s['summary']}"
        net.add_node(
            s['id'],
            label=label,
            title=title_html,
            color={
                'background': color + '33',  # transparent fill
                'border': color,
                'highlight': {'background': color + '55', 'border': '#222'},
            },
            font={'color': '#222', 'size': 12},
        )

    # Add edges
    for s in steps:
        for dep in s.get('depends_on', []):
            is_back = dep > s['id']
            net.add_edge(
                dep, s['id'],
                color={'color': '#e74c3c' if is_back else '#999'},
                dashes=is_back,
            )

    # Legend HTML
    legend_html = '<div style="margin:12px 0;display:flex;gap:14px;flex-wrap:wrap;font-family:-apple-system,sans-serif;font-size:12px">'
    for t, c in TYPE_COLORS.items():
        legend_html += f'<div style="display:flex;align-items:center;gap:6px"><span style="width:14px;height:14px;background:{c};border-radius:3px;border:1px solid #ccc"></span><span style="color:#555">{t}</span></div>'
    legend_html += '</div>'

    # Render
    html = net.generate_html(notebook=True)
    full = f'<div style="font-family:-apple-system,sans-serif;margin:16px 0"><div style="font-size:15px;font-weight:600;color:#222;margin-bottom:8px">{title}</div>{legend_html}{html}</div>'
    display(HTML(full))


print('Helpers defined.')

## Math problem — Bayesian probability

We start with a classic Bayesian probability question that requires setting up the theorem, computing intermediate probabilities, and applying the formula. A reasoning model should produce a chain of observations and deductions leading to a clean conclusion.

In [ ]:
MATH_PROBLEM = """
A company has 3 servers. Server A handles 50% of requests, B handles 30%, C handles 20%.
Error rates: A=2%, B=5%, C=10%.
A request fails. What is the probability it came from Server C?
"""

print('Step 1: Generating reasoning trace...')
math_result = call_and_reason(MATH_PROBLEM)
print(f"Done. {math_result['usage']['completion_tokens']} tokens.\n")
print('=== REASONING TRACE ===')
print(math_result['trace'])
print('\n=== FINAL ANSWER ===')
print(math_result['answer'])

Once we have the trace, we extract its structure. The extraction call returns a JSON array — each item is one step with its type, summary, and list of step ids it depends on.

In [ ]:
print('Step 2: Extracting structured steps...')
math_steps = extract_steps(math_result['trace'])
print(f'{len(math_steps)} steps extracted.\n')
print_steps(math_steps)

Now we visualize the reasoning as a directed graph. Nodes are colored by step type, edges show logical dependencies. The graph is interactive — drag nodes to rearrange, scroll to zoom, hover for details.

In [ ]:
print('Step 3: Drawing reasoning graph...')
draw_graph(math_steps, title='Reasoning Graph — Bayesian Probability')

## Code problem — Bug detection

Next, a Python function with a subtle off-by-one bug. This kind of problem benefits from step-by-step tracing: the model needs to walk through what the code actually does on small inputs and identify where the output diverges from the expected behavior.

In [ ]:
CODE_PROBLEM = """
Find and fix the bug in this Python function:

```python
def fibonacci(n):
    if n <= 0:
        return 0
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return a
```

Expected: fibonacci(1)=1, fibonacci(6)=8, fibonacci(10)=55
"""

print('Step 1: Generating reasoning trace...')
code_result = call_and_reason(CODE_PROBLEM)
print(f"Done. {code_result['usage']['completion_tokens']} tokens.\n")
print('=== REASONING TRACE ===')
print(code_result['trace'])
print('\n=== FINAL ANSWER ===')
print(code_result['answer'])

Same extraction step — we ask the model to surface the structure of its own reasoning.

In [ ]:
print('Step 2: Extracting structured steps...')
code_steps = extract_steps(code_result['trace'])
print(f'{len(code_steps)} steps extracted.\n')
print_steps(code_steps)

Bug-detection reasoning typically has a clear linear shape: observe the symptom, hypothesize the cause, trace through the failing case, verify, and propose a fix. The graph makes this trajectory visible.

In [ ]:
print('Step 3: Drawing reasoning graph...')
draw_graph(code_steps, title='Reasoning Graph — Bug Detection')

## Abstract problem — Open-ended reasoning

Finally, a conceptual question with no single correct answer. Abstract problems push the model into a different reasoning mode: instead of applying a formula or tracing code, it must weigh arguments, draw on multiple frames, and synthesize a position.

In [ ]:
ABSTRACT_PROBLEM = """
Why does trust take a long time to build but can be destroyed in an instant?
"""

print('Step 1: Generating reasoning trace...')
abstract_result = call_and_reason(ABSTRACT_PROBLEM)
print(f"Done. {abstract_result['usage']['completion_tokens']} tokens.\n")
print('=== REASONING TRACE ===')
print(abstract_result['trace'])
print('\n=== FINAL ANSWER ===')
print(abstract_result['answer'])

In [ ]:
print('Step 2: Extracting structured steps...')
abstract_steps = extract_steps(abstract_result['trace'])
print(f'{len(abstract_steps)} steps extracted.\n')
print_steps(abstract_steps)

Abstract reasoning often produces wider, more branching graphs than math or code — multiple parallel hypotheses converge into a single conclusion. This is where the visualization is most useful, since the structure is genuinely hard to follow as plain text.

In [ ]:
print('Step 3: Drawing reasoning graph...')
draw_graph(abstract_steps, title='Reasoning Graph — Abstract Problem')

## Next steps

You now have a reusable pipeline to turn any model reasoning into a structured graph. A few directions to explore from here:

* Swap in [Magistral](https://docs.mistral.ai/capabilities/reasoning/native) (`magistral-small-latest` or `magistral-medium-latest`) to work with native thinking traces instead of prompted ones — Magistral returns its reasoning in a dedicated `thinking` field without needing the labeling system prompt.
* Add custom step types (e.g. `assumption`, `counter-example`) by editing the system prompt and the color map.
* Export graphs as standalone HTML with `net.save_graph('graph.html')` to share auditable reasoning traces outside the notebook.
* Run the pipeline on a batch of problems and compare graph topology (depth, branching factor) across problem types as a proxy for reasoning complexity.